In [1]:
"""
Separation-baseline closure for a cord-bound pair — symbolic consistency battery.

Construction under test
-----------------------
    eta      = R_sA R_sB / (R_sA + R_sB)^2                 Horizon Symmetry (R.O.M.)
    x_A      = R_sB / (R_sA + R_sB) ,  x_B = R_sA / (R_sA + R_sB)
               x_A + x_B = 1 ,  x_A x_B = eta
    r_AC     = x_A d ,  r_BC = x_B d
    beta_sep = beta_AC + beta_BC = beta_AC / x_A
    kappa_sep^2       = (R_sA + R_sB) / d
    closure           : kappa_sep^2 + kappa_cord^2 = 2 beta_sep^2
    effective sep     : d_eff = (R_sA + R_sB) / (2 beta_sep^2)
    cord tension      : T = eta M c^2 kappa_cord^2 / (2 d)

Masses appear only as a bridge for checking against the classical two-body
result; no test below depends on them as primitives.

Run:  python globes_construction_tests.py
"""
import sympy as sp

mA, mB, d, G, c, w, vr = sp.symbols('m_A m_B d G c omega v_rel', positive=True)

RsA, RsB = 2*G*mA/c**2, 2*G*mB/c**2
M        = mA + mB
eta      = sp.simplify(RsA*RsB/(RsA + RsB)**2)
xA       = sp.simplify(RsB/(RsA + RsB))
xB       = sp.simplify(RsA/(RsA + RsB))
rAC, rBC = xA*d, xB*d
k_sep2   = (RsA + RsB)/d

bAC, bBC = w*rAC/c, w*rBC/c
b_sep    = sp.simplify(bAC + bBC)

w_free   = sp.sqrt(G*M/d**3)
b_sep_f  = w_free*d/c
T_free   = 2*sp.pi/w_free
d_eff    = sp.simplify((RsA + RsB)/(2*b_sep_f**2))

vrel2    = sp.Symbol('v_rel2', positive=True)
v_closed = sp.solve(sp.Eq(k_sep2, 2*vrel2/c**2), vrel2)[0]

T_A = mA*w**2*rAC - G*mA*mB/d**2
T_B = mB*w**2*rBC - G*mA*mB/d**2
mu  = mA*mB/M

d_eff_s, r_effA = sp.symbols('d_eff r_effA', positive=True)
bAC_eff = sp.simplify((xA**2*(RsA + RsB)/(2*d_eff_s)).subs(d_eff_s, r_effA/xA))

CHECKS = {
    "eta = mu/M":                              eta - mu/M,
    "x_A + x_B = 1":                           xA + xB - 1,
    "x_A x_B = eta":                           xA*xB - eta,
    "x_A is the barycentre split":             xA - mB/M,
    "beta_sep = beta_AC / x_A":                b_sep - bAC/xA,
    "beta_sep = omega d / c":                  b_sep - w*d/c,
    "free pair: v_rel^2 = GM/d":               v_closed - G*M/d,
    "slack cord: d_eff = d":                   d_eff - d,
    "Kepler: R_sA+R_sB = 8pi^2 d^3/(T^2c^2)":  (RsA + RsB) - 8*sp.pi**2*d**3/(T_free**2*c**2),
    "scale recovery = 1 at every q":           8*sp.pi**2*d**3/(T_free**2*c**2)/(RsA + RsB) - 1,
    "kappa_cord^2 = kappa_sep^2 e":            (2*b_sep**2 - k_sep2) - k_sep2*(2*b_sep**2/k_sep2 - 1),
    "e = v_rel^2 d/(GM) - 1":                  (2*(vr/c)**2/k_sep2 - 1) - (vr**2*d/(G*M) - 1),
    "one cord, one tension: T_A = T_B":        T_A - T_B,
    "T = eta M c^2 kappa_cord^2 / (2d)":       T_A - mu*c**2*(2*b_sep**2 - k_sep2)/(2*d),
}

CONSEQUENCES = {
    "kappa_sep^2 / kappa_AC^2":  sp.simplify(k_sep2/(RsA/rAC)),       # = R_sB/R_sA
    "beta_AC^2 via r_eff,A":     sp.factor(sp.simplify(bAC_eff)),
    "  at equal scales":         sp.simplify(bAC_eff.subs(mB, mA)),
    "  section as written":      sp.simplify(RsA/(2*r_effA)),
}

if __name__ == "__main__":
    bad = 0
    for name, expr in CHECKS.items():
        ok = sp.simplify(expr) == 0
        bad += not ok
        print(f"{'PASS' if ok else 'FAIL':4}  {name}" + ("" if ok else f"   residual {sp.simplify(expr)}"))
    print()
    for name, expr in CONSEQUENCES.items():
        print(f"{name:28} {expr}")
    print(f"\n{len(CHECKS)-bad}/{len(CHECKS)} checks pass")

PASS  eta = mu/M
PASS  x_A + x_B = 1
PASS  x_A x_B = eta
PASS  x_A is the barycentre split
PASS  beta_sep = beta_AC / x_A
PASS  beta_sep = omega d / c
PASS  free pair: v_rel^2 = GM/d
PASS  slack cord: d_eff = d
PASS  Kepler: R_sA+R_sB = 8pi^2 d^3/(T^2c^2)
PASS  scale recovery = 1 at every q
PASS  kappa_cord^2 = kappa_sep^2 e
PASS  e = v_rel^2 d/(GM) - 1
PASS  one cord, one tension: T_A = T_B
PASS  T = eta M c^2 kappa_cord^2 / (2d)

kappa_sep^2 / kappa_AC^2     m_B/m_A
beta_AC^2 via r_eff,A        G*m_B**3/(c**2*r_effA*(m_A + m_B)**2)
  at equal scales            G*m_A/(4*c**2*r_effA)
  section as written         G*m_A/(c**2*r_effA)

14/14 checks pass
